# Using the PhenoMe pipeline with STED-FM models and data
## Optim dataset
---

## 1. Setup

In [1]:
import torch
from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

### Wrap the STED-FM model

In [2]:
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper

class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1, # For single channel images
            as_classifier=True,
            # global_pool = "patch" # Returns the embeddings for all patches
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

wrapper = STEDFMWrapper(device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### Using DINOv2

In [13]:
wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

Using cache found in /Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/renaud/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


### Custom transforms for Optim dataset

In [3]:
import torchvision.transforms as T

# Transforms for STED-FM model
stedfm_transforms = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224), interpolation=0),
    T.Grayscale(num_output_channels=1),
    T.ToTensor(),
])

# Transforms for STED-FM model
stedfm_transforms = T.Compose([T.ToTensor()])

In [16]:
from skimage import io

img = io.imread("/Users/renaud/datasets/optim-dataset/images/img_3.tif")
print(img.shape)
print(img.min(), img.max())

(224, 224)
0.0 1.0


### Get the Optim dataset

In [ ]:
import os

# Download dataset

home = "~/.stedfm"

!mkdir -p {home}/evaluation-data
if not os.path.isfile(os.path.join(home, "evaluation-data", "optim-data.zip")):
    #!wget -O {home}/evaluation-data/optim-data.zip https://s3.valeria.science/flclab-foundation-models/evaluation-data/optim-data.zip
    !curl -L -o {home}/evaluation-data/optim-data.zip https://s3.valeria.science/flclab-foundation-models/evaluation-data/optim-data.zip

if not os.path.isdir(os.path.join(home, "evaluation-data", "optim-data")):
    !unzip {home}/evaluation-data/optim-data.zip -d {home}/evaluation-data

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  132M  100  132M    0     0  28.6M      0  0:00:04  0:00:04 --:--:-- 28.6M


In [6]:
from stedfm.datasets import OptimDataset
from stedfm.loaders import get_dataset

# Restructure dataset for phenome usage
data_path = "/Users/renaud/datasets/optim-data/optim-data"

_, _, test_loader = get_dataset(name="optim")

/Users/renaud/.stedfm/evaluation-data/optim_train
Train dataset size: 1536 --> (array([0, 1, 2, 3]), array([881, 220, 208, 227]))
Valid dataset size: 147 --> (array([0, 1, 2, 3]), array([82, 29, 16, 20]))
Test dataset size: 438 --> (array([0, 1, 2, 3]), array([261,  60,  56,  61]))


In [8]:
# Iterate through dataset and save individual images, masks and metadata.csv
import os

import numpy as np
import pandas as pd
import tifffile
from tqdm import tqdm

tiles_dir = "/Users/renaud/datasets/optim-dataset"
tiles_images_dir = os.path.join(tiles_dir, "images")
tiles_metadata_path = os.path.join(tiles_dir, "metadata.csv")

labels = ['actin', 'tubulin', 'CaMKII', 'PSD95']

os.makedirs(tiles_images_dir, exist_ok=True)

rows = []
for idx, (image, meta) in enumerate(tqdm(test_loader, desc="Saving tiles")):
    stem = (
        f"img_{idx}"
    )

    image = np.asarray(image, dtype=np.float32)
    while image.ndim > 2:
        image = image[0]

    tifffile.imwrite(os.path.join(tiles_images_dir, f"{stem}.tif"), image)
    row_dict = {k: meta.get(k).item() for k in meta.keys()}
    row_dict["label"] = labels[row_dict["label"]]
    row_dict["filename"] = stem
    rows.append(row_dict)

# 'filename' is the extension-less stem shared by the image and its mask, which is what
# make_dataframe_metadata_fn matches on in pheno.find_files().
tiles_metadata_df = pd.DataFrame(rows)
tiles_metadata_df.to_csv(tiles_metadata_path, index=False)

print(f"Saved {len(tiles_metadata_df)} tiles to {tiles_images_dir}")
print(f"Metadata written to {tiles_metadata_path}")
tiles_metadata_df.head()

Saving tiles: 100%|██████████| 438/438 [00:33<00:00, 12.96it/s] 

Saved 438 tiles to /Users/renaud/datasets/optim-dataset/images
Metadata written to /Users/renaud/datasets/optim-dataset/metadata.csv


,label,dataset-idx,score,filename
0,actin,218,0.844,img_0
1,actin,169,0.825,img_1
2,CaMKII,331,0.765,img_2
3,actin,98,0.776,img_3
4,PSD95,414,0.726,img_4


### Create dataset (find files)

In [22]:
# csv-based metadata
from phenome import make_dataframe_metadata_fn
from data_handling import minmax_preprocessing_fn
import pandas as pd

images_dir = "/Users/renaud/datasets/optim-dataset/images"
metadata_dir = "/Users/renaud/datasets/optim-dataset/metadata.csv"

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=None, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")



Found 438 files total.
DataFrame: 438 files, 7 columns.


Found 438 images


### Process images

In [36]:
# Process with STED-FM model
checkpoint_path = 'checkpoints/optim_dataset/optim_v2_stedfm.h5'

pheno.process_images(wrapper, 
                    force_rgb=False,
                    checkpoint_path=checkpoint_path,
                    preprocessing_fn=None,
                    custom_transformations=stedfm_transforms,
                    l2_normalize_channels=False
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Processing batches:   0%|          | 0/14 [00:00<?, ?it/s]

Pipeline ready: 438 images available (lazy loading from checkpoints/optim_dataset/optim_v2_stedfm.h5).


Computing properties:   0%|          | 0/438 [00:00<?, ?img/s]

Checkpoint: saved 438 properties (refreshed).


,image_index,image_path,image_name,intensity_min,sharpness_metric,intensity_mean,intensity_entropy,intensity_max,intensity_std
0,0,/Users/renaud/datasets/optim-dataset/images/im...,img_0.tif,0,0.275598,0.050424,3.755296,1,0.079212
1,1,/Users/renaud/datasets/optim-dataset/images/im...,img_1.tif,0,0.264506,0.040069,2.888038,1,0.082970
2,2,/Users/renaud/datasets/optim-dataset/images/im...,img_10.tif,0,0.267108,0.054349,3.668250,1,0.092000
3,3,/Users/renaud/datasets/optim-dataset/images/im...,img_100.tif,0,0.358040,0.061956,6.042111,1,0.064390
4,4,/Users/renaud/datasets/optim-dataset/images/im...,img_101.tif,0,0.232677,0.040729,2.695590,1,0.069090
...,...,...,...,...,...,...,...,...,...
433,433,/Users/renaud/datasets/optim-dataset/images/im...,img_95.tif,0,0.282850,0.048363,3.839375,1,0.072179
434,434,/Users/renaud/datasets/optim-dataset/images/im...,img_96.tif,0,0.309922,0.092604,4.505223,1,0.106250
435,435,/Users/renaud/datasets/optim-dataset/images/im...,img_97.tif,0,0.057381,0.006984,0.280255,1,0.036240
436,436,/Users/renaud/datasets/optim-dataset/images/im...,img_98.tif,0,0.306787,0.049130,4.324903,1,0.063532


In [15]:
# Process with DINO-v2 model

pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path='checkpoints/optim_dataset/optim_dinov2.h5',
                    )
pheno.compute_properties(property_preset="complete")

Processing batches:   0%|          | 0/14 [00:00<?, ?it/s]

Pipeline ready: 438 images available (lazy loading from checkpoints/optim_dataset/optim_dinov2.h5).


Computing properties:   0%|          | 0/438 [00:00<?, ?img/s]

,image_index,image_path,image_name,intensity_min,intensity_max,intensity_std,intensity_mean,sharpness_metric,intensity_entropy
0,0,/Users/renaud/datasets/optim-dataset/images/im...,img_0.tif,0,1,0.079212,0.050424,0.275598,3.755296
1,1,/Users/renaud/datasets/optim-dataset/images/im...,img_1.tif,0,1,0.082970,0.040069,0.264506,2.888038
2,2,/Users/renaud/datasets/optim-dataset/images/im...,img_10.tif,0,1,0.092000,0.054349,0.267108,3.668250
3,3,/Users/renaud/datasets/optim-dataset/images/im...,img_100.tif,0,1,0.064390,0.061956,0.358040,6.042111
4,4,/Users/renaud/datasets/optim-dataset/images/im...,img_101.tif,0,1,0.069090,0.040729,0.232677,2.695590
...,...,...,...,...,...,...,...,...,...
433,433,/Users/renaud/datasets/optim-dataset/images/im...,img_95.tif,0,1,0.072179,0.048363,0.282850,3.839375
434,434,/Users/renaud/datasets/optim-dataset/images/im...,img_96.tif,0,1,0.106250,0.092604,0.309922,4.505223
435,435,/Users/renaud/datasets/optim-dataset/images/im...,img_97.tif,0,1,0.036240,0.006984,0.057381,0.280255
436,436,/Users/renaud/datasets/optim-dataset/images/im...,img_98.tif,0,1,0.063532,0.049130,0.306787,4.324903


## 4. Cluster in embedding space

Clusters are **data-driven** groups in embedding space; they may or may not match your experimental labels. Set `n_clusters` between 2 and the number of images (see [Pipeline API](../../docs/src/content/docs/advanced/api/pipeline.md)). After clustering, labels are stored under metadata key `cluster`.

In [38]:
pheno.compute_clustering(n_clusters=3, clustering_method="dbscan", dbscan_eps=0.5)
print(pheno.get_available_property_keys())
print(pheno.get_available_metadata_keys())

pheno.plot_tsne(color_by="label")
pheno.plot_tsne(color_by="cluster")

Clustering: PCA extracted 100 components from 384 dimensions before DBSCAN.
Clustering (dbscan): 1 clusters on 438 images (source=embeddings).


['intensity_entropy', 'intensity_max', 'intensity_mean', 'intensity_min', 'intensity_std', 'sharpness_metric']
['cluster', 'dataset-idx', 'filename', 'id', 'label', 'mask_path', 'score']


## Interactive Exploration

The pipeline provides an interactive dashboard using Plotly and ipywidgets. This allows you to explore the embedding space, highligh specific groups, and inspect individual images by clicking on dots.

Note: This requires a Jupyter environment with `ipywidgets` and `plotly` installed.


In [39]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

In [42]:
print(pheno.get_embeddings())

[[ 0.4940979  -0.15642206  0.16755281 ... -0.08746281 -0.10659971
   0.23227245]
 [ 0.7412068   0.02284756  0.02199288 ... -0.32124972 -0.21679543
   0.02871149]
 [ 0.4910155  -0.0863172   0.04681274 ... -0.15222144 -0.22366065
   0.14079086]
 ...
 [ 1.4690845  -0.09965071  0.05177866 ... -0.35084644 -0.10247457
   0.23225403]
 [ 0.4877159  -0.07567067  0.07629384 ... -0.06640187 -0.14330094
   0.09132167]
 [ 0.91226655 -0.07476225  0.24811381 ... -0.43762866 -0.24789537
   0.11671644]]
